# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the dataset URLurl = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(url)metadata = dataset.metadataprint(f"{metadata.name}: {metadata.description}")print(f"License: {metadata.license}")print(f"Published: {metadata.datePublished}")

## 2. Data OverviewReview available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their @id, and corresponding field IDsprint("Available RecordSets and their Fields:\n")record_sets = metadata.record_setsfor rs in record_sets:    print(f"RecordSet name: {rs.name}")    print(f"  @id: {rs.id}")    print("  Fields:")    for field in rs.fields:        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")    print()

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the first record set for extractionfirst_record_set_id = metadata.record_sets[0].id# For demonstration, show all idsall_record_set_ids = [rs.id for rs in metadata.record_sets]dataframes = {}for record_set_id in all_record_set_ids:    records = list(dataset.records(record_set=record_set_id))    dataframes[record_set_id] = pd.DataFrame(records)print(f"Columns for RecordSet {first_record_set_id}:")print(dataframes[first_record_set_id].columns.tolist())dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field (column) from the first RecordSet for analysis# For demonstration, let's auto-select the first numeric field (if any)df = dataframes[first_record_set_id]numeric_field = None# Find a likely numeric fieldfor field in metadata.record_sets[0].fields:    # Heuristic: if type is Float or Integer or has number in type    if ('Float' in field.data_type) or ('Integer' in field.data_type) or ('Number' in str(field.data_type)) or ('Double' in field.data_type):        if field.id in df.columns:            numeric_field = field.id            print(f"Chose numeric field: {field.name} (@id: {field.id})")            breakif numeric_field is None:    raise ValueError("No numeric field found in the first record set.")# Try to cast to float if not already numericdf[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')threshold = df[numeric_field].quantile(0.75)  # use 75th percentilefiltered_df = df[df[numeric_field] > threshold]print(f"Filtered records with {numeric_field} > {threshold}:")print(filtered_df.head())filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()print(f"Normalized {numeric_field} for filtered records:")print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())# Choose a group field (string/categorical) for groupinggroup_field = Nonefor field in metadata.record_sets[0].fields:    # Heuristic: type Text or String, but not the numeric field    if field.id != numeric_field and (str(field.data_type).lower().find('text') != -1 or str(field.data_type).lower().find('string') != -1):        if field.id in df.columns:            group_field = field.id            print(f"Group by field: {field.name} (@id: {field.id})")            breakif group_field:    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")    print(f"Grouped data by {group_field} (mean of {numeric_field}):")    print(grouped_df.head())

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snsplt.figure(figsize=(8,4))sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)plt.title(f"Distribution of {numeric_field}")plt.xlabel(numeric_field)plt.show()if group_field:    plt.figure(figsize=(10,4))    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])    plt.title(f"Boxplot of {numeric_field} by {group_field}")    plt.xlabel(group_field)    plt.ylabel(numeric_field)    plt.xticks(rotation=45)    plt.show()

## 6. ConclusionSummarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the dataset via its Croissant schema.- Fields and data were referenced using their `@id`s, ensuring traceability.- Numeric fields were filtered and normalized; group-level aggregations provided basic insight.- Simple distribution and categorical visualizations enabled a quick quality assessment of the data.- You can extend the analysis by exploring other RecordSets, fields, or constructing advanced analyses pertinent to mass spectrometry protein data.